# 싱글턴 vs 멀티턴 호출 차이

Groq Chat Completions에서 독립 호출과 메시지 히스토리 기반 대화 호출의 차이를 확인합니다.


## 0. 준비


In [1]:
import os

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 GROQ_API_KEY를 먼저 설정하세요.")

MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
client = Groq(api_key=api_key)

print(f"사용 모델: {MODEL}")


사용 모델: llama-3.1-8b-instant


In [2]:
def safe_create(label: str, messages: list[dict], **kwargs):
    print(f"\n--- {label} ---")
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            **kwargs,
        )
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        print(f"Groq API 오류: {status_code} {type(exc).__name__}")
        print(exc)
        return None

    print(response.choices[0].message.content)
    if getattr(response, "usage", None):
        print(
            f"\n토큰: prompt={response.usage.prompt_tokens}, "
            f"completion={response.usage.completion_tokens}, total={response.usage.total_tokens}"
        )
    return response


## 1. 싱글턴 호출

각 요청이 독립적으로 처리됩니다. 두 번째 요청은 첫 번째 응답을 자동으로 기억하지 않습니다.


In [3]:
single_first = safe_create(
    "싱글턴 1",
    messages=[{"role": "user", "content": "파이썬이 뭐야? 한 문장으로 답해줘."}],
    max_completion_tokens=120,
)



--- 싱글턴 1 ---
파이썬은 간결하고 읽기 쉬운 프로그래밍 언어로, 데이터 분석, 웹 개발, 인공지능, 게임 프로그래밍 등 다양한 분야에서 사용되는 고급 프로그래밍 언어입니다.

토큰: prompt=53, completion=51, total=104


In [4]:
single_second = safe_create(
    "싱글턴 2",
    messages=[{"role": "user", "content": "방금 말한 것보다 더 쉽게 설명해줘."}],
    max_completion_tokens=120,
)



--- 싱글턴 2 ---
하지만 이전에 어떤 말을 했다는 기록이 없소. 처음으로 질문하시는 분이시겠네요. 무슨 내용에 대해 질문하거나 설명 부탁드립니다.

토큰: prompt=49, completion=35, total=84


## 2. 메시지 히스토리로 멀티턴 호출

Groq Chat Completions는 이전 대화를 `messages` 배열에 직접 넣어 대화를 이어갑니다.


In [5]:
history = [
    {
        "role": "system",
        "content": "당신은 초보자를 가르치는 Python 튜터입니다. 짧고 친절하게 답하세요.",
    }
]


def chat_turn(user_message: str, max_completion_tokens: int = 180):
    history.append({"role": "user", "content": user_message})
    response = safe_create(
        f"멀티턴: {user_message}",
        messages=history,
        max_completion_tokens=max_completion_tokens,
    )
    if response is not None:
        history.append(
            {
                "role": "assistant",
                "content": response.choices[0].message.content,
            }
        )
    return response


chat_turn("파이썬이 뭐야? 한 문장으로 답해줘.", 120)



--- 멀티턴: 파이썬이 뭐야? 한 문장으로 답해줘. ---
파이썬(Python)은 코드를 쉽게 작성할 수 있는 프로그래밍 언어로, 게임, 웹, 데이터 분석, 운영 체제 등 다양한 곳에서 사용됩니다.

토큰: prompt=77, completion=41, total=118


ChatCompletion(id='chatcmpl-94ac13c1-46ad-4309-8e2e-e875c188ea8e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='파이썬(Python)은 코드를 쉽게 작성할 수 있는 프로그래밍 언어로, 게임, 웹, 데이터 분석, 운영 체제 등 다양한 곳에서 사용됩니다.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055433, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_7ccc667439', usage=CompletionUsage(completion_tokens=41, prompt_tokens=77, total_tokens=118, completion_time=0.073215014, completion_tokens_details=None, prompt_time=0.004651065, prompt_tokens_details=None, queue_time=0.006576703, total_time=0.077866079), usage_breakdown=None, x_groq=XGroq(id='req_01kwk65fn6f8t92zw1qw3zygry', debug=None, seed=645430493, usage=None))

In [6]:
chat_turn("방금 말한 것보다 더 쉽게 설명해줘.", 160)



--- 멀티턴: 방금 말한 것보다 더 쉽게 설명해줘. ---
파이썬은 프로그래밍을 쉽고 즐겁게 도와주는 컴퓨터 언어입니다.

토큰: prompt=141, completion=26, total=167


ChatCompletion(id='chatcmpl-2403d40b-057d-49b1-85c9-d48e6bcaac50', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='파이썬은 프로그래밍을 쉽고 즐겁게 도와주는 컴퓨터 언어입니다.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055433, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_7ccc667439', usage=CompletionUsage(completion_tokens=26, prompt_tokens=141, total_tokens=167, completion_time=0.040731926, completion_tokens_details=None, prompt_time=0.014441566, prompt_tokens_details=None, queue_time=0.003426264, total_time=0.055173492), usage_breakdown=None, x_groq=XGroq(id='req_01kwk65fyver3atkj11x474fs6', debug=None, seed=1647313270, usage=None))

In [7]:
chat_turn("예시 코드도 짧게 줘.", 220)



--- 멀티턴: 예시 코드도 짧게 줘. ---
파이썬에서는 간단한 로그인이 가능합니다.

```python
print("Hello, World!")  # 화면에 "Hello, World!" 출력
```

이 코드를 실행하면, "Hello, World!"가 화면에 나타납니다.

토큰: prompt=186, completion=58, total=244


ChatCompletion(id='chatcmpl-e758f130-2fa2-442a-8c7c-1cc440045333', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='파이썬에서는 간단한 로그인이 가능합니다.\n\n```python\nprint("Hello, World!")  # 화면에 "Hello, World!" 출력\n```\n\n이 코드를 실행하면, "Hello, World!"가 화면에 나타납니다.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055434, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_7ccc667439', usage=CompletionUsage(completion_tokens=58, prompt_tokens=186, total_tokens=244, completion_time=0.095011536, completion_tokens_details=None, prompt_time=0.011553889, prompt_tokens_details=None, queue_time=0.007122331, total_time=0.106565425), usage_breakdown=None, x_groq=XGroq(id='req_01kwk65g7zebavq4vyp82c8wgs', debug=None, seed=78249774, usage=None))

In [8]:
print("--- 현재 history 대화 기록 ---")
for index, message in enumerate(history, start=1):
    content = message["content"].replace("\n", " ")
    print(f"{index}. {message['role']}: {content[:120]}")


--- 현재 history 대화 기록 ---
1. system: 당신은 초보자를 가르치는 Python 튜터입니다. 짧고 친절하게 답하세요.
2. user: 파이썬이 뭐야? 한 문장으로 답해줘.
3. assistant: 파이썬(Python)은 코드를 쉽게 작성할 수 있는 프로그래밍 언어로, 게임, 웹, 데이터 분석, 운영 체제 등 다양한 곳에서 사용됩니다.
4. user: 방금 말한 것보다 더 쉽게 설명해줘.
5. assistant: 파이썬은 프로그래밍을 쉽고 즐겁게 도와주는 컴퓨터 언어입니다.
6. user: 예시 코드도 짧게 줘.
7. assistant: 파이썬에서는 간단한 로그인이 가능합니다.  ```python print("Hello, World!")  # 화면에 "Hello, World!" 출력 ```  이 코드를 실행하면, "Hello, World!"가 화면에


## 3. 시스템 메시지 + 멀티턴


In [9]:
tutor_history = [
    {
        "role": "system",
        "content": (
            "당신은 초보자를 가르치는 Python 튜터입니다. "
            "답변은 쉬운 비유, 짧은 코드, 확인 질문 순서로 작성하세요."
        ),
    }
]


def tutor_turn(user_message: str):
    tutor_history.append({"role": "user", "content": user_message})
    response = safe_create(
        f"튜터 대화: {user_message}",
        messages=tutor_history,
        max_completion_tokens=240,
    )
    if response is not None:
        tutor_history.append({"role": "assistant", "content": response.choices[0].message.content})
    return response


tutor_turn("리스트가 뭐야?")
tutor_turn("딕셔너리랑 차이가 뭔데?")



--- 튜터 대화: 리스트가 뭐야? ---
네, 리스트는 물건을 나란히 담는 상자와 비슷합니다. 리스트 상자는 다양한 물건(변수)를 담을 수 있습니다. 하나의 상자 안에 여러 가지 물건을 넣을 수 있잖아요? 

Python에서 리스트를 만드는 방법은 다음과 같습니다:

```python
주먹공치 = ['사과', '배', '감정']
```

여기서 `주먹공치`는 이름, `'사과'`, `'배'`, `'감선'`이 리스트상자 속에 담겨 있습니다. 

리스트 상자안에 담긴 물건을 하나씩 꺼내고 싶으면 인덱스를 사용해줘야 합니다. 

이런 물건들을 하나씩 하나씩 담을 때, 1, 2, 3... 인덱스를 대체로 사용할 거로 예상해 볼 수 있을 겁니다. 

위 예에서 `주먹공치[0]`를 작성하면 `"사과"`가 나옵니다.

토큰: prompt=76, completion=223, total=299

--- 튜터 대화: 딕셔너리랑 차이가 뭔데? ---
딕셔너리와 리스트는 모두 데이터를 담는 상자입니다. 차이점이 있습니다. 

리스트는 물건을 나란히 담는 상자와 비슷하다면,
딕셔너리는 물건을 라벨로 열거하는 상자라고 생각할 수 있습니다. 

한 장난감 상자에는 "사과"를 "apple", "배"를 "banana", "감정"을 "grape" 이라고 적어서 라벨을 걸 수도 있습니다. 

Python에서 딕셔너리를 만드는 방법은 다음과 같습니다:

```python
주먹공치 = {'사과': 'apple', '배': 'banana', '감정': 'grape'}
```

여기서 `주먹공치`라는 이름에 `'사과'`와 `'배'`, `'감정'`을 라벨로 적어놨는데, 그리고 그 라벨에 일치하는 데이터를 담았습니다. 

딕셔너리에서 "사과"를 꺼내고 싶으면 `주먹공치['사과']` 같은 문장을 사용할 수

토큰: prompt=320, completion=240, total=560


ChatCompletion(id='chatcmpl-813ef939-5856-4408-96d2-01db77894197', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='딕셔너리와 리스트는 모두 데이터를 담는 상자입니다. 차이점이 있습니다. \n\n리스트는 물건을 나란히 담는 상자와 비슷하다면,\n딕셔너리는 물건을 라벨로 열거하는 상자라고 생각할 수 있습니다. \n\n한 장난감 상자에는 "사과"를 "apple", "배"를 "banana", "감정"을 "grape" 이라고 적어서 라벨을 걸 수도 있습니다. \n\nPython에서 딕셔너리를 만드는 방법은 다음과 같습니다:\n\n```python\n주먹공치 = {\'사과\': \'apple\', \'배\': \'banana\', \'감정\': \'grape\'}\n```\n\n여기서 `주먹공치`라는 이름에 `\'사과\'`와 `\'배\'`, `\'감정\'`을 라벨로 적어놨는데, 그리고 그 라벨에 일치하는 데이터를 담았습니다. \n\n딕셔너리에서 "사과"를 꺼내고 싶으면 `주먹공치[\'사과\']` 같은 문장을 사용할 수', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055435, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_7ccc667439', usage=CompletionUsage(completion_tokens=240, prompt_tokens=320, total_tokens=560, completion_time=0.40256

## 4. 기록 길이와 토큰 비용

메시지 히스토리를 계속 추가하면 모델이 참고할 문맥은 늘어나지만, 매 요청의 입력 토큰도 함께 증가합니다.


In [10]:
long_history = [
    {"role": "system", "content": "짧고 명확하게 답하세요."}
]

questions = [
    "API 키를 안전하게 보관하는 방법을 한 문장으로 알려줘.",
    "방금 답변에서 가장 중요한 규칙 하나만 골라줘.",
    "그 규칙을 어겼을 때 생기는 문제를 예시로 들어줘.",
]

for question in questions:
    long_history.append({"role": "user", "content": question})
    response = safe_create(
        f"컨텍스트 성장: {question}",
        messages=long_history,
        max_completion_tokens=160,
    )
    if response is None:
        break
    long_history.append({"role": "assistant", "content": response.choices[0].message.content})
    print(f"현재 메시지 수: {len(long_history)}")



--- 컨텍스트 성장: API 키를 안전하게 보관하는 방법을 한 문장으로 알려줘. ---
API 키를 안전하게 보관하려면 환경 변수(Environment Variable)를 사용하거나 비밀 보관소(KeyCloak Secret Management이나 Hashicorp Vault 등) 또는 암호화를 통해 서버 또는 클라이언트에 저장하지 말고 암호화된 형식으로 보관하는 것이 좋습니다.

토큰: prompt=62, completion=64, total=126
현재 메시지 수: 3

--- 컨텍스트 성장: 방금 답변에서 가장 중요한 규칙 하나만 골라줘. ---
환경 변수나 비밀 보관소 사용 없이 API 키를 서버 또는 클라이언트에 저장하지 마세요.

토큰: prompt=150, completion=25, total=175
현재 메시지 수: 5

--- 컨텍스트 성장: 그 규칙을 어겼을 때 생기는 문제를 예시로 들어줘. ---
API 키를 서버에 저장했다고 가정할 때, 만약 API의 비밀 키를 탈취하거나 위협이 들어간 경우, 서버가 보유한 API 키를 통해서 제 3자나 공격자가 사용자가 비밀 키로 로그인한 것 처럼 모든 권한을 획득할 수 있습니다.

토큰: prompt=203, completion=71, total=274
현재 메시지 수: 7


## 5. 스트리밍 응답

긴 답변을 기다리지 않고 생성되는 텍스트를 조금씩 받을 수 있습니다.


In [11]:
def stream_groq(label: str, messages: list[dict], **kwargs):
    print(f"\n--- {label} ---")
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            stream=True,
            **kwargs,
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                print(delta, end="", flush=True)
        print()
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        print(f"Groq API 오류: {status_code} {type(exc).__name__}")
        print(exc)


In [12]:
stream_groq(
    "스트리밍 단일 응답",
    messages=[
        {"role": "user", "content": "Groq 스트리밍이 필요한 상황을 3가지로 설명해줘."}
    ],
    max_completion_tokens=260,
)



--- 스트리밍 단일 응답 ---
Groq 스트리밍은 딥러닝 inference를 위한 고성능 가속기 기술입니다. Groq는 GPU, TPU, 그리고 중앙 처리 장치(CPU)와 같이 일반적인 AI 가속 장치를 제외한 고유한 아키텍처를 활용합니다. Groq 스트리밍이 필요한 3 가지 상황은 다음과 같습니다.

1. **대규모 모델 inference**: Groq 스트리밍은 매우 큰 모델의 inference를 위한 최적화된 솔루션입니다. 이러한 모델은 일반적으로 높은 추론 비용과 관련이 있습니다. Groq 스트리밍을 사용하면 모델 추론을 가속화하고 수요가 많은 inference 환경에서 고성능을 제공할 수 있습니다.

2. **스트리밍 환경**: Groq 스트리밍은 스트리밍 인지 모델(예: 추천 시스템, 자막 인식 등)이 필요한 상황에서 설계되었습니다. Groq 스트리밍을 사용하면 이러한 모델을 스트리밍 인프라에서 쉽게 실행할 수 있습니다.

3. **비즈니스 크리티컬한 시스템**: 이러한 시스템은 종종 실시간 성을 요구하는 상황에서 고성능을 제공해야 할 수 있습니다. Groq 스트


In [13]:
stream_groq(
    "스트리밍 멀티턴 응답",
    messages=history + [{"role": "user", "content": "지금까지의 설명을 한 문단으로 요약해줘."}],
    max_completion_tokens=180,
)



--- 스트리밍 멀티턴 응답 ---
파이썬은 프로그래밍을 쉽고 즐겁게 도와주는 컴퓨터 언어입니다. 예를들어 파이썬으로 "Hello, World!"이라는 메세지를 화면에 출력할 수 있습니다. (print("Hello, World!")과 같은 쉬운 코드를 통해 파이썬의 유용성을 직접 경험할 수 있습니다.)


## 6. 정리

| 방식 | 장점 | 주의점 |
| --- | --- | --- |
| 싱글턴 | 단순하고 빠름 | 이전 대화를 기억하지 않음 |
| 메시지 히스토리 | 대화 맥락 유지 | 매 요청마다 입력 토큰 증가 |
| 시스템 메시지 | 역할과 규칙 고정 | 너무 긴 규칙은 비용 증가 |
| 스트리밍 | 빠른 체감 응답 | 이벤트 처리 코드 필요 |
